<a href="https://colab.research.google.com/github/Musamehar/ML_Intership/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question & Business Decision

*   **Research Question:** How accurately can we predict and rank organic content items at risk of performance decay using historical search visibility, engagement, and content metadata?
*   **Decision Supported:** Helps SEO strategists and editorial teams allocate limited weekly bandwidth to perform content refresh audits on pages facing traffic loss before rankings drop significantly.
*   **Unit of Analysis:** One row = One unique content item (`content_id`) for a specific client (`client_id`).
*   **Cost of Error:**
    *   *False Positive:* Wasted editorial hours auditing healthy pages.
    *   *False Negative:* Unnoticed traffic decay leading to compounded loss in organic search reach.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Setup repository and dataset path
if not os.path.exists('ML_Intership') and not Path('data/raw/content_refresh_anonymized.csv').exists():
    !git clone https://github.com/Musamehar/ML_Intership.git

possible_paths = [
    Path('ML_Intership/data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv')
]

data_path = next((p for p in possible_paths if p.exists()), None)
df = pd.read_csv(data_path)

# Filter qualified slice per FlyRank rules
df_clean = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()

print(f"Dataset Scale: {len(df_clean):,} qualified mature content items across {df_clean['client_id'].nunique()} unique client domains.")

Cloning into 'ML_Intership'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 134 (delta 44), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.87 MiB | 15.16 MiB/s, done.
Resolving deltas: 100% (44/44), done.
Dataset Scale: 30,000 qualified mature content items across 32 unique client domains.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data Source, Window, and Safety Exclusions

*   **Dataset Release:** FlyRank ML Internship anonymized search warehouse dataset.
*   **Time Window:** Trailing 90-day feature observation window predicting subsequent organic performance decay.
*   **Deliberate Exclusions (Leakage Guard):**
    1.  `trend_pct` and `trend_direction` are strictly excluded from all feature sets because the decay label (`is_declining_label`) is directly derived from them.
    2.  Client identifiers (`client_id`) are used strictly for grouped cross-validation splits and are never passed as predictive features.
*   **Public Safety:** No real URLs, domain names, client names, or private search queries are included.

In [2]:
# Define target label
if 'is_declining_label' in df_clean.columns:
    df_clean['target'] = df_clean['is_declining_label'].astype(int)
else:
    df_clean['target'] = (df_clean['trend_direction'] == 'down').astype(int)

# Feature engineering (pre-decision knowable signals only)
df_clean['log_impressions_90d'] = np.log1p(df_clean['impressions_90d'])
df_clean['log_clicks_90d'] = np.log1p(df_clean['clicks_90d'])
df_clean['ctr_90d'] = df_clean['clicks_90d'] / (df_clean['impressions_90d'] + 1e-5)
df_clean['has_keyword'] = (df_clean['word_count'] > 0).astype(int)

feature_cols = ['log_impressions_90d', 'log_clicks_90d', 'ctr_90d', 'avg_position', 'content_age_days', 'has_keyword']
X = df_clean[feature_cols].fillna(0)
y = df_clean['target']
groups = df_clean['client_id']

print("Data safety check completed:")
print(f"Feature matrix shape: {X.shape}")
print(f"Decay target positive base rate: {y.mean()*100:.2f}% ({y.sum():,} positive cases)")

Data safety check completed:
Feature matrix shape: (30000, 6)
Decay target positive base rate: 54.21% (16,262 positive cases)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology & Validation Strategy

1.  **Baseline Rule:** Transparent heuristic score (0.0 – 100.0) combining normalized log impressions (50%), content age $\ge 180$ days (35%), and top-10 SERP positioning (15%).
2.  **Machine Learning Model:** Random Forest Classifier (`n_estimators=100`, `max_depth=8`) modeling non-linear interactions between staleness, search volume, and SERP position.
3.  **Honest Validation Design:** 5-Fold `GroupKFold` split grouped strictly by `client_id` to ensure models are evaluated on unseen client domains and prevent domain leakage.

In [3]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Compute Week 4 Rule Baseline Score
log_imp = df_clean['log_impressions_90d']
vis_score = (log_imp - log_imp.min()) / (log_imp.max() - log_imp.min())
age_risk = np.where(df_clean['content_age_days'] >= 180, 1.0, df_clean['content_age_days'] / 180.0)
pos_opp = np.where((df_clean['avg_position'] > 0) & (df_clean['avg_position'] <= 10), 1.0, 0.5)
df_clean['baseline_score'] = (0.50 * vis_score + 0.35 * age_risk + 0.15 * pos_opp) * 100.0

gkf = GroupKFold(n_splits=5)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
lr_model = LogisticRegression(max_iter=1000, random_state=42)

print("Methodology setup complete: Models and GroupKFold cross-validation initialized.")

Methodology setup complete: Models and GroupKFold cross-validation initialized.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Honest Comparison Table
We compare the Rule Baseline, Logistic Regression, and Random Forest models across identical 5-fold grouped splits using **ROC-AUC**, **Average Precision (AP)**, and **Precision@20**.

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score

models = {
    'Rule Baseline (W04)': None,
    'Logistic Regression': lr_model,
    'Random Forest': rf_model
}

results = []

for name, model in models.items():
    auc_scores, ap_scores, p20_scores = [], [], []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if name == 'Rule Baseline (W04)':
            preds = df_clean.iloc[val_idx]['baseline_score'] / 100.0
        else:
            model.fit(X_train, y_train)
            preds = model.predict_proba(X_val)[:, 1]

        auc_scores.append(roc_auc_score(y_val, preds))
        ap_scores.append(average_precision_score(y_val, preds))

        val_df = pd.DataFrame({'y_true': y_val, 'pred': preds})
        top20 = val_df.sort_values(by='pred', ascending=False).head(20)
        p20_scores.append(top20['y_true'].mean())

    results.append({
        'Model': name,
        'ROC-AUC': np.mean(auc_scores),
        'Avg Precision': np.mean(ap_scores),
        'Precision@20': np.mean(p20_scores)
    })

results_df = pd.DataFrame(results).round(4)
print("=" * 65)
print("CAPSTONE RESULTS: MODEL VS. BASELINE (5-FOLD GROUPED SPLIT)")
print("=" * 65)
print(results_df.to_string(index=False))
print("=" * 65)

CAPSTONE RESULTS: MODEL VS. BASELINE (5-FOLD GROUPED SPLIT)
              Model  ROC-AUC  Avg Precision  Precision@20
Rule Baseline (W04)   0.5844         0.5904          0.49
Logistic Regression   0.6752         0.6927          0.83
      Random Forest   0.6695         0.6779          0.65


## 5. Limitations

*What this work cannot claim.*

### What This Work Cannot Claim

*   **Directional Decision Support Only:** This system provides prioritization scores to guide human review; it does not claim to predict Google's ranking algorithms directly or guarantee traffic recovery.
*   **Unbalanced Tracking History:** Newer client accounts in the panel lack extended historical tracking, requiring missing tracking filters.
*   **External Confounders:** The model cannot anticipate macro SERP layout changes (such as Google adding AI Overviews) or seasonal demand fluctuations.

In [5]:
# Error case extraction for qualitative inspection
rf_model.fit(X, y)
df_clean['rf_prob'] = rf_model.predict_proba(X)[:, 1]

fp_cases = df_clean[(df_clean['target'] == 0) & (df_clean['rf_prob'] > 0.70)].head(3)
fn_cases = df_clean[(df_clean['target'] == 1) & (df_clean['rf_prob'] < 0.30)].head(3)

print("False Positive Cases (High predicted risk, actual healthy):")
print(fp_cases[['content_id', 'impressions_90d', 'avg_position', 'content_age_days', 'rf_prob']].to_string(index=False))

print("\nFalse Negative Cases (Low predicted risk, actual declining):")
print(fn_cases[['content_id', 'impressions_90d', 'avg_position', 'content_age_days', 'rf_prob']].to_string(index=False))

False Positive Cases (High predicted risk, actual healthy):
          content_id  impressions_90d  avg_position  content_age_days  rf_prob
content_9d548144b06d               86          12.6               118 0.717460
content_55f75c034970             3998           6.4               140 0.772741
content_42f79b19d0e4             3063           5.8               165 0.714933

False Negative Cases (Low predicted risk, actual declining):
          content_id  impressions_90d  avg_position  content_age_days  rf_prob
content_d8a23b5e10c5                2           7.5               126 0.229132
content_b382d571a4d4              518          51.4               463 0.247847
content_474bc8a4a3cb                3           5.7               173 0.266101


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Action Playbook Output
The output ranks mature pages requiring review and attaches specific reason codes:
*   `stale_visible_decay`: High impression visibility ($\ge 500$) with content age $\ge 180$ days.
*   `page_one_decay_risk`: Position 1–10 ranking pages facing staleness.

In [6]:
def assign_reason_code(row):
    if row['content_age_days'] >= 180 and row['impressions_90d'] >= 500:
        return 'stale_visible_decay'
    elif 0 < row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        return 'page_one_decay_risk'
    else:
        return 'general_refresh_candidate'

df_clean['reason_code'] = df_clean.apply(assign_reason_code, axis=1)
df_clean['action_label'] = 'content_refresh_audit'

ranked_queue = df_clean.sort_values(by='rf_prob', ascending=False).reset_index(drop=True)
output_cols = ['content_id', 'client_id', 'rf_prob', 'reason_code', 'action_label', 'impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days']

top_queue = ranked_queue[output_cols].head(10)
print("TOP 10 RANKED RECOMMENDATIONS FOR EDITORIAL REVIEWS:")
print(top_queue.to_string(index=False))

TOP 10 RANKED RECOMMENDATIONS FOR EDITORIAL REVIEWS:
          content_id         client_id  rf_prob               reason_code          action_label  impressions_90d  clicks_90d  avg_position  content_age_days
content_9ef8e61550e9 client_9400f1b21c 0.818687       stale_visible_decay content_refresh_audit             2812           1           8.8               557
content_78d9e6444e78 client_9400f1b21c 0.816776       stale_visible_decay content_refresh_audit             3499           1           7.9               557
content_0be51c9e6cbd client_f369cb89fc 0.814383 general_refresh_candidate content_refresh_audit             4205           2           0.7               106
content_8e32f425a490 client_9400f1b21c 0.812415       stale_visible_decay content_refresh_audit             1767           1           8.7               557
content_64b2508ad098 client_9400f1b21c 0.812297       stale_visible_decay content_refresh_audit             1531           1           9.2               557
conte

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Generating Paper Artifacts
Export metric summary CSVs and charts to `work/outputs/` for embedding directly into the published research paper.

In [7]:
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('../work/outputs', exist_ok=True)

# 1. Save results metric table
results_df.to_csv('work/outputs/model_vs_baseline_results.csv', index=False)

# 2. Save Feature Importance plot
plt.figure(figsize=(8, 4))
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()
importances.plot(kind='barh', color='#2b5c8f')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Weight')
plt.tight_layout()
plt.savefig('work/outputs/feature_importance.png', dpi=300)
plt.close()

# 3. Save Ranked Recommendations CSV
ranked_queue[output_cols].to_csv('work/outputs/capstone_ranked_recommendations.csv', index=False)

print("SUCCESS: Generated and saved all capstone paper artifacts in work/outputs/.")

SUCCESS: Generated and saved all capstone paper artifacts in work/outputs/.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.